# TUTORIAL: Bias-aware data assimilation on the Rijke tube


In [ ]:
from romda.utils import set_working_directories
data_folder, results_folder = set_working_directories('rijke/')[:2]


In [ ]:
from romda.models.physical import Rijke
from romda.observations import Observations

truth = Observations(model=Rijke,
                     manual_bias='linear',
                     t_start=.2,
                     t_stop=.6,
                     Nt_obs=20)

truth.plot_truth(truth, f_max=1200, window=0.02, fig_width=12)

In [ ]:
from romda.ensemble import Ensemble
from romda.data_assimilation import rBA_EnKF  # try also EnKF / EnSRKF


ensemble = Ensemble(parent_model=Rijke(dt=truth.dt),
                    da_method=rBA_EnKF,
                    regularization_factor=1.,
                    m=10,
                    std_phi=0.25,
                    std_alpha=dict(beta=[3., 4.],
                                   tau=[1e-3, 2e-3]),
                    distribution_alpha='uniform',
                    inflation_factor=1.0,
                    inflation_factor_rejection=1.005,
                    )

ensemble.visualize_state()

In [ ]:
from romda.bias_estimators import ESN_bias
import numpy as np

training_data_filename = f"{results_folder}/ESN_train_data_rijke_{truth.name_bias}"

bias_estimator = ESN_bias(rom=ensemble.model,
                          reference_data=truth,
                          training_data_filename=training_data_filename,
                          # Training data generation options
                          augment_data=True,
                          biased_observations=False,
                          correlation_based_training=False,
                          L=20,
                          std_alpha=0.3,
                          # Training, validation and test times
                          t_val=0.02,
                          t_train=0.25,
                          t_test=0.06,
                          N_units=100,
                          N_wash=15,
                          upsample=2,
                          # Hyperparameter search ranges
                          rho_range=(0.2, 1.0),
                          tikh_range=[1e-16],
                          sigma_in_range=(np.log10(1e-5), np.log10(1e-2)),
                          plot_training=True,
                          N_ens=10,
                          )

In [ ]:
filter_ens = ensemble.copy()
filter_ens.bias = bias_estimator.copy()
filter_ens.regularization_factor = 1.

# Observation error covariance matrix
std_obs = 0.1
Cdd = np.diag(std_obs * np.ones(filter_ens.model.Nq)) * np.max(abs(truth.y_obs), axis=0) ** 2

for d, t_d in zip(truth.y_obs, truth.t_obs):
    filter_ens.forecast_step(t_end=t_d)
    filter_ens.analysis_step(d=d, Cdd=Cdd.copy())

# Forecast beyond the last observation and close the multiprocessing pools
filter_ens.forecast_step(t_end=truth.t_obs[-1] + 10 * filter_ens.model.t_CR, close=True)

In [ ]:
filter_ens.visualize_history(truth=truth, plot_members=True, dims=[0, 1])
filter_ens.visualize_history(truth=truth, plot_members=False, reference_a=truth.true_parameters)

In [ ]:
# Visualize the estimated bias and innovations over the assimilation window
filter_ens.bias.visualize_bias_and_innovations(plot_members=True)